In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway_coherence = os.path.join(pathway_temp, "uher2008coherence")
pathway_personality = os.path.join(pathway_temp, "uher2008personality")
original_data_pathway = os.path.join(pathway_coherence, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "uher2008coherence_observations.csv")
complete_path_2 = os.path.join(original_data_pathway, "uher2008coherence_test.csv")

out_coherence_pathway = os.path.join(pathway_coherence, "standardized_data")
if not os.path.exists(out_coherence_pathway):
    os.makedirs(out_coherence_pathway)

out_personality_pathway = os.path.join(pathway_personality, "standardized_data")
if not os.path.exists(out_personality_pathway):
    os.makedirs(out_personality_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)


In [3]:

df1.rename(columns={"species": "species_original",
    "sub": "participant",
    "day":"day_original"}, inplace=True)

# df1['date'] = df1['date'].astype(str).str[:-3]
##for some reason separate this one below - but none of these work. Asked Steven
# df1['date'] = pd.to_datetime(df1['date'],unit='s')
# df1['date'].unique()
# df1.columns

In [4]:
observation_dates = [
       ['bonobo_t1_afternoon_observations', '02',''],
       ['bonobo_t2_afternoon_obeservations', '02','03'],
       ['chimpanzee_t1_afternoon_observations', '02','03'],
       ['chimpanzee_t2_afternoon_obeservations', '02','03'],
       ['gorilla_t1_afternoon_observations','02',''],
      ['gorilla_t2_afternoon_obeservations', '02','03'],
       ['orangutan_t1_afternoon_observations','02','03'],
       ['orangutan_t2_afternoon_obeservations', '02', '03']]
for x,y,a in observation_dates:
       df1.loc[df1.data_subset_name == x, ['month', 'day']] = y, a

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "uher2008coherence_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df1['participant'] = df1['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df1['participant'].replace(x, y, inplace=True)
# names =df1['ape'].unique().tolist()
# ape_study_working = pd.DataFrame(names)
# ape_study_working.to_csv('ape_study_working.csv', encoding='utf-8-sig', index=False)
ape=[]
for index, row in df2.iterrows():
    if not pd.isna(row['sub']):
        ape.append(row['sub'])
    elif not pd.isna(row['subject']):
        ape.append(row['subject'])
    else:
        ape.append("")
df2 = df2.assign(participant=ape)

df2['participant'] = df2['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df2['participant'].replace(x, y, inplace=True)

In [6]:
df2_5 = df2[df2['data_subset_name'] == 'food_competition_test']
df2_5 = df2_5.dropna(axis=1, how='all')
df2 = df2[df2['data_subset_name'] != 'food_competition_test']

df2_5.rename(columns={"opponent": "participant_2"}, inplace=True)
comp_path_name_errors = os.path.join(pathway_gen, "uher2008coherence_name_errors.csv")

df2_5['participant_2'] = df2_5['participant_2'].str.rstrip()
df_name  = pd.read_csv(comp_path_name_errors)
df2_5['participant_2'] = df2_5['participant_2'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df2_5['participant_2'].replace(x, y, inplace=True)

df2_5 = df2_5.assign(role='focal_participant')
df2_5 = df2_5.assign(role_2='opponent')
df2_5['dyad']=df2_5.participant.str.cat(df2_5.participant_2, sep='_')

In [7]:
df2_6 = df2[df2['data_subset_name'] == 'hidden_food_test']
df2_6 = df2_6.dropna(axis=1, how='all')
df2 = df2[df2['data_subset_name'] != 'hidden_food_test']

df2_6.rename(columns={"trial": "food_item"}, inplace=True)

In [8]:
data_frames=[df1, df2, df2_5, df2_6]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    data_frames[index]=x
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [9]:

fulldf = fulldf.assign(year='2005')

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='participant_2', right_on='name_2', how='left')

In [10]:
complete_path_age = os.path.join(original_data_pathway, "table1.csv")
add_age = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(add_age,left_on='participant', right_on='name', how='left')

In [11]:
fulldf.rename(columns={"species_y": "species",
    'touch_1':	'touched_box',
    'open_1':	'opened_box',
    'eat_1'	:'item_eaten_or_tasted',
    'reject_1'	:'food_rejected',
    'l_touch':	'latency_to_touch_box',
    'l_open':	'latency_to_open_box',
    'l_t60':	'not_touching',
    'l_o60':	'not_opening',
    'fr_1': 	'duration_of_proximity_to_experimenter', 
    'ag_1': 	'freq_of_quasi-aggressive_acts', 
    'ax_1': 	'food_taken', 
    'occ_1': 	'duration_occupied', 
    'reach_1': 	'reaching_frequency', 
    'temp_1': 	'knocking_frequency', 
    'gain_1': 	'successful_reaching',
    'enter_1': 	'entered_or_not',
    'fight_1': 	'fight_with_fingers_in_box',
    'lat_1': 	'latency_to_reach_or_touch',
    'lat_6001': 	'not_found_or_taken',
    'total_1': 	'percentage_items_found', 
    'reg_1': 	'seconds_on_tape_for_latency', 
    'hon_1': 	'duration',
    's_ag_1': 	'severe_direct_attacks', 
    'init_1': 	'climbing_up_stake', 
    'pilo_1': 	'pilo_erection', 
    'lat_60_1':	'latency_60',
    'rej_1':	'rejecting_food',
    'kind_nfo':	'novel_food_type', 
    'sort': 	'food_type', 
    'obj_1':	'duration_of_action_with_object',
    'temper':	'frequency_of_knocking',
    'reach':	'reaching',
    'rock_1':	'rocking',
    'scrat_1':	'scratching',
    'grin_1':	'grinning',
    'voc_1':	'vocalizing',
    'pace_1':	'pacing',
    'locat_1':	'changing_location',
    'wrist_1':	'wrist_shaking',	
    'list_1':	'listening',
    'sex_1':	'sexual_activity',
    'vigi':	'vigilance',
    'prox':	'proximity_to_other',
    'conta':	'bodily_contact_with_1',
    'give_gro':	'gives_grooming_to',
    'rec_gro':	'receives_grooming_from',
    's_groom':	'self_grooming',
    's_scra':	'self_scratch_1',
    's_care':	'self_care',
    'nurse':	'nursing_child',
    'al_play':	'allo_play',
    'au_play':	'auto_play_with_object',
    'obj_play':	'object_play',
    'al_sex':	'allo_sex',
    'au_sex':	'auto_sex',
    'posi':	'position_1',
    'au_groom':'auto_grooming',
    'al_groom':'allo_grooming',
    'activ':	'active',
    's_scratc':	'self_scratch_2',
    "data_subset_name":"experiment_name",
    'in_out':'experimenter_in_or_out'
    }, inplace=True)
# column_name = fulldf.columns.values.tolist()
# print(column_name)

In [12]:

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['proximity_to_other'].replace(x, y, inplace=True)
    fulldf['bodily_contact_with_1'].replace(x, y, inplace=True)
    fulldf['contact'].replace(x, y, inplace=True)
    fulldf['nursing_child'].replace(x, y, inplace=True)
    fulldf['gives_grooming_to'].replace(x, y, inplace=True)
    fulldf['receives_grooming_from'].replace(x, y, inplace=True)
    fulldf['allo_play'].replace(x, y, inplace=True)
    fulldf['allo_sex'].replace(x, y, inplace=True)

In [13]:
fulldf.dropna(subset=['participant'], inplace=True)

fulldf.rename(columns={'study_id':'study_id_personality'}, inplace=True)
fulldf = fulldf.assign(study_id='uher2008coherence')

In [14]:
experiment_paths = os.path.join(original_data_pathway, "uher_experiment_names.csv")
exp_df = pd.read_csv(experiment_paths)
fulldf= fulldf.merge(exp_df,left_on='experiment_name', right_on='name_1', how='left')

In [15]:
fulldf['experiment'].unique()
fulldf['experiment'] = fulldf['experiment'].astype(str)


In [16]:
spe_2=[] 
for index, row in fulldf.iterrows():
    if not pd.isna(row['bodily_contact_with_1']):
        spe_2.append(row['bodily_contact_with_1'])
    else:
        spe_2.append(row['contact'])
fulldf = fulldf.assign(bodily_contact_with=spe_2)

spe_3=[] 
for index, row in fulldf.iterrows():
    if not pd.isna(row['self_scratch_1']):
        spe_3.append(row['self_scratch_1'])
    else:
        spe_3.append(row['self_scratch_2'])
fulldf = fulldf.assign(self_scratch=spe_3)
# fulldf.columns
fulldf.rename(columns={"age": "age_in_years"}, inplace=True)


In [17]:
fulldf=fulldf[[ 'study_id', 'experiment',  'experiment_name','year','month','day',
       'participant',   'age_in_years','sex', 'role',
       'participant_2','opponent',  'sex_2', 'role_2', 'dyad', 'species', 
       'session', 'trial', 're_trial','period','experimenter_in_or_out',  'unit', 'time',
        'rocking', 'scratching', 'grinning', 'vocalizing', 'pacing', 'wrist_shaking', 'changing_location', 'sexual_activity',
        
        'vigilance','o_vigi',  'proximity_to_other','o_prox',  'bodily_contact_with','contact', 'o_cont', 
        'gives_grooming_to', 'allo_grooming','o_al_gr', 
        'receives_grooming_from','auto_grooming','o_au_gr',  'self_grooming',
         'self_care','o_s_car', 'self_scratch', 'o_s_scr', 
         'allo_play','o_al_pl',  'auto_play_with_object','o_au_pl',  'object_play', 'o_obj_pl',
        'allo_sex', 'o_al_sex',  'auto_sex','o_au_sex',  'feed','o_feed', 
        'nursing_child','o_nurs', 'activity','active', 'position','position_1',

          'adultnea', 'yasa',

          'duration_occupied', 'duration_of_proximity_to_experimenter', 'freq_of_quasi-aggressive_acts', 'food_taken',  
          'touched_box', 'opened_box', 'item_eaten_or_tasted', 'food_rejected', 'latency_to_touch_box', 
          'latency_to_open_box', 'not_touching', 'not_opening', 'reaching_frequency', 'knocking_frequency', 
          
          'successful_reaching', 'entered_or_not', 'fight_with_fingers_in_box', 'latency_to_reach_or_touch', 'not_found_or_taken', 'percentage_items_found', 
          'seconds_on_tape_for_latency', 'duration', 'severe_direct_attacks', 
          'climbing_up_stake', 'pilo_erection', 'food_type', 'novel_food_type', 
          'food', 'rejecting_food', 'latency_60', 'duration_of_action_with_object', 
          'reaching', 'frequency_of_knocking',  'listening','food_item',
          'notes1', 'notes2', ]]


In [18]:

for index in range(1,15):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_coherence_path = os.path.join(out_coherence_pathway, 'uher2008coherence_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_coherence_path, encoding='utf-8-sig', index=False)
    ##uherpersonality
    exp['study_id'].replace('uher2008coherence', 'uher2008personality', inplace=True, regex=True)
    comp_out_personality_path = os.path.join(out_personality_pathway, 'uher2008personality_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_personality_path, encoding='utf-8-sig', index=False)
    ##glossaries
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_coherence_path_glossary = os.path.join(out_coherence_pathway, 'uher2008coherence_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_coherence_path_glossary, encoding='utf-8-sig', index=False)
    ##uherpersonality
    comp_out_personality_path_glossary = os.path.join(out_personality_pathway, 'uher2008personality_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_personality_path_glossary, encoding='utf-8-sig', index=False)


In [19]:
obs1 = fulldf[fulldf['experiment'] == '0']
obs1 = obs1.dropna(axis=1, how='all')
obs1.rename(columns={"activity": "activity_temp",
                     "auto_grooming":"auto_grooming_temp"}, inplace=True)

grooming=[]
for index, row in obs1.iterrows():
    if not pd.isna(row['auto_grooming_temp']):
        grooming.append(row['auto_grooming_temp'])
    elif not pd.isna(row['self_grooming']):
        grooming.append(row['self_grooming'])
    else:
        grooming.append(np.nan)
obs1 = obs1.assign(auto_grooming=grooming)

activity=[]
for index, row in obs1.iterrows():
    if not pd.isna(row['activity_temp']):
        activity.append(row['activity_temp'])
    elif not pd.isna(row['active']):
        activity.append(row['active'])
    else:
        activity.append(np.nan)
obs1 = obs1.assign(activity=activity)
obs1.rename(columns={"o_vigi": "o_vigilance",
                     "o_prox":"o_proximity_to_other",
                     "o_cont":"o_bodily_contact_with",
                     "o_al_gr":"o_allo_grooming",
                     "o_au_gr":"o_auto_grooming",
                     "o_s_car":"o_self_care",
                     "o_s_scr":"o_self_scratch",
                     "o_al_pl":"o_allo_play",
                     "o_au_pl":"o_auto_play",
                     "o_obj_pl":"o_object_play",
                     "o_al_sex":"o_allo_sex",
                     "o_au_sex":"o_auto_sex",
                     "o_nurs":"o_nursing_child",
                     'unit':'trial'}, inplace=True)
obs1.columns

obs1['notes']=obs1.notes1.str.cat(obs1.notes2, sep=' ')
obs1['notes'].replace(' ', '_', inplace=True, regex=True)


In [20]:

obs1 = obs1[['study_id', 'experiment', 'experiment_name', 'year', 'month', 'day',
       'participant', 'age_in_years', 'sex', 'species','trial', 're_trial', 'period',
        'time', 'rocking', 'scratching', 'grinning', 'vocalizing',
       'pacing', 'wrist_shaking', 'changing_location', 'sexual_activity',
       'vigilance', 'o_vigilance', 'proximity_to_other',
       'o_proximity_to_other', 'bodily_contact_with', 'contact',
       'o_bodily_contact_with', 'gives_grooming_to', 'allo_grooming',
       'o_allo_grooming', 'receives_grooming_from', 
       'auto_grooming', 
       'o_auto_grooming',  'self_care', 'o_self_care',
       'self_scratch', 'o_self_scratch', 'allo_play', 'o_allo_play',
       'auto_play_with_object', 'o_auto_play', 'object_play', 'o_object_play',
       'allo_sex', 'o_allo_sex', 'auto_sex', 'o_auto_sex', 'feed', 'o_feed',
       'nursing_child', 'o_nursing_child', 'activity','position',
       'position_1', 'notes']]

In [21]:
# obs_temp = obs1[['study_id', 'experiment', 'experiment_name', 'year', 'month','day',
#        'participant', 'age_in_years','sex', 'species', 're_trial', 'period',  't',
#        'unit', 'time', 
#        'rocking', 'scratching', 'grinning', 'vocalizing',
#        'pacing', 'wrist_shaking', 'changing_location', 'sexual_activity',
#        'vigilance',  'proximity_to_other',
#        'bodily_contact_with','contact', 'gives_grooming_to',
#        'allo_grooming',  'receives_grooming_from', 'auto_grooming','self_grooming', 'self_care', 
#         'self_scratch',  
#         'allo_play', 'auto_play_with_object', 'object_play', 
#        'allo_sex',  'auto_sex', 'feed', 
#        'nursing_child',  
       # 'activity',  
       # 'position',
       # 'adult', 'adultnea', 'yasa',
       # 'notes1', 'notes2']].values.tolist() + obs1[[
       #  'study_id', 'experiment', 'experiment_name', 'year', 'month','day',
       # 'participant', 'age_in_years','sex', 'species', 're_trial', 'period', 't',
       # 'unit', 'time', 
       # 'rocking', 'scratching', 'grinning', 'vocalizing',
       # 'pacing', 'wrist_shaking', 'changing_location', 'sexual_activity',
       # 'o_vigi', 'proximity_to_other', 
       #  'o_cont','contact', 'gives_grooming_to',
       #  'o_al_gr', 'receives_grooming_from', 'o_au_gr', 'self_grooming',  'o_s_car', 
       #  'o_s_scr',
       #   'o_al_pl', 'o_au_pl',  'o_obj_pl',
       #  'o_al_sex', 'o_au_sex', 'o_feed',
       #  'o_nurs',
       #   'active',
       #    'position_1',
       # 'adult', 'adultnea', 'yasa',
       # 'notes1', 'notes2'
       # ]].values.tolist()

# obs1 = pd.DataFrame(obs_temp, columns=['study_id', 'experiment', 'experiment_name', 'year', 'month','day',
#        'participant', 'age_in_years', 'sex', 'species', 're_trial', 'period',  't',
#        'unit', 'time', 
#        'rocking', 'scratching', 'grinning', 'vocalizing',
#        'pacing', 'wrist_shaking', 'changing_location', 'sexual_activity',
#        'vigilance',  'proximity_to_other',
#        'bodily_contact_with', 'contact', 'gives_grooming_to',
#        'allo_grooming',  'receives_grooming_from', 'auto_grooming','self_grooming', 'self_care', 
#         'self_scratch', 
#         'allo_play', 'auto_play_with_object', 'object_play', 
#        'allo_sex',  'auto_sex', 'feed', 
#        'nursing_child',  
#        'activity',  
#        'position',
#        'adult', 'adultnea', 'yasa',
       # 'notes1', 'notes2' ])

In [22]:

obs1['experiment_name'].replace("obeservations", "observations", inplace=True, regex=True)
# obs1['experiment_name'].unique()

obs1.loc[obs1.experiment_name == 'gorilla_t1_afternoon_observations', ['contact']] = np.nan

In [23]:
prefeeding = ['bonobo_prefeeding_observations',
       'chimpanzee_prefeeding_observations',
       'gorilla_prefeeding_observations',
       'orangutan_prefeeding_observations']
afternoon = ['bonobo_t1_afternoon_observations',
       'bonobo_t2_afternoon_observations',
       'chimpanzee_t1_afternoon_observations',
       'chimpanzee_t2_afternoon_observations',
       'gorilla_t1_afternoon_observations',
       'gorilla_t2_afternoon_observations',
       'orangutan_t1_afternoon_observations',
       'orangutan_t2_afternoon_observations']

obs2 = obs1[~obs1['experiment_name'].isin(afternoon)]
obs2 = obs2.dropna(axis=1, how='all')
obs3 = obs1[~obs1['experiment_name'].isin(prefeeding)]
obs3 = obs3.dropna(axis=1, how='all')

In [24]:
comp_out_coherence_path_stand = os.path.join(out_coherence_pathway, 'uher2008coherence_observations_prefeeding_standardized.csv')
obs2.to_csv(comp_out_coherence_path_stand, encoding='utf-8-sig', index=False)

obs2['study_id'].replace('uher2008coherence', 'uher2008personality', inplace=True, regex=True)
comp_out_personality_path_stand = os.path.join(out_personality_pathway, 'uher2008personality_observations_prefeeding_standardized.csv')
obs2.to_csv(comp_out_personality_path_stand, encoding='utf-8-sig', index=False)

comp_out_coherence_path_stand = os.path.join(out_coherence_pathway, 'uher2008coherence_observations_afternoon_standardized.csv')
obs3.to_csv(comp_out_coherence_path_stand, encoding='utf-8-sig', index=False)

obs3['study_id'].replace('uher2008coherence', 'uher2008personality', inplace=True, regex=True)
comp_out_personality_path_stand = os.path.join(out_personality_pathway, 'uher2008personality_observations_afternoon_standardized.csv')
obs3.to_csv(comp_out_personality_path_stand, encoding='utf-8-sig', index=False)

In [25]:
names = obs2.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
uher2008coherence_uher2008personality_observations_glossary=df[["column_name", "description"]]

comp_out_coherence_path_glossary = os.path.join(out_coherence_pathway, 'uher2008coherence_observations_prefeeding_glossary.csv')
uher2008coherence_uher2008personality_observations_glossary.to_csv(comp_out_coherence_path_glossary, encoding='utf-8-sig', index=False)

comp_out_personality_path_glossary = os.path.join(out_personality_pathway, 'uher2008personality_observations_prefeeding_glossary.csv')
uher2008coherence_uher2008personality_observations_glossary.to_csv(comp_out_personality_path_glossary, encoding='utf-8-sig', index=False)


In [26]:
names = obs3.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
uher2008coherence_uher2008personality_observations_glossary=df[["column_name", "description"]]

comp_out_coherence_path_glossary = os.path.join(out_coherence_pathway, 'uher2008coherence_observations_afternoon_glossary.csv')
uher2008coherence_uher2008personality_observations_glossary.to_csv(comp_out_coherence_path_glossary, encoding='utf-8-sig', index=False)

comp_out_personality_path_glossary = os.path.join(out_personality_pathway, 'uher2008personality_observations_afternoon_glossary.csv')
uher2008coherence_uher2008personality_observations_glossary.to_csv(comp_out_personality_path_glossary, encoding='utf-8-sig', index=False)

In [27]:
##experiment 5 option2 

# complete_path_3 = os.path.join(out_coherence_pathway, "uher2008coherence_exp5_standardized.csv")
# df_exp5 = pd.read_csv(complete_path_3)

# df_exp5=df_exp5.sort_values(by = ['participant','session','trial'])
# # df_exp5.columns

In [28]:
dyad_lists = [['joey_kuno','kuno_joey',],
              ['joey_limbuko','limbuko_joey',],
              ['joey_yasa','yasa_joey',],
              ['kuno_limbuko','limbuko_kuno',],
              ['kuno_yasa','yasa_kuno',],
              ['limbuko_yasa','yasa_limbuko',],
              ['frodo_fraukje','fraukje_frodo',],
              ['frodo_sandra','sandra_frodo',],
              ['frodo_robert','robert_frodo',],
              ['fraukje_sandra','sandra_fraukje',],
              ['fraukje_robert','robert_fraukje',],
              ['robert_sandra', 'sandra_robert',],
              ['bebe_n’diki','n’diki_bebe',],
              ["bebe_ruby_b'windi", "ruby_b'windi_bebe",],
              ['bebe_viringika','viringika_bebe',],
              [ "n’diki_ruby_b'windi", "ruby_b'windi_n’diki",],
              ['n’diki_viringika','viringika_n’diki',],
              ["ruby_b'windi_viringika","viringika_ruby_b'windi", ],
              ['dokana_dunja','dunja_dokana',],
              ['dokana_padana','padana_dokana',],
              ['dokana_pini','pini_dokana',],
              ['dunja_padana','padana_dunja',],
              ['dunja_pini','pini_dunja',],
              ['padana_pini','pini_padana',]]

# test_list = []
# for x,y in dyad_lists:
#     dyad_1 = df_exp5[df_exp5.dyad.str.contains(x)] 
#     dyad_2 = df_exp5[df_exp5.dyad.str.contains(y)] 
#     dyad_1_list =dyad_1[['study_id', 'experiment', 'experiment_name', 'year', 'participant',
#        'sex', 'role',  'dyad', 'species', 'session', 'trial', 'period', 'gain', 'enter', 'fight', 'latency']].values.tolist() 
#     dyad_2_list = dyad_2[[ 'participant',
#        'sex', 'role', 'dyad', 'session', 'trial', 'gain', 'enter', 'fight', 'latency']].values.tolist() 
#     combined_lol = [lol_1+lol_2 for lol_1,lol_2 in zip(dyad_1_list,dyad_2_list)]
#     test_list = test_list + combined_lol

# df_exp5_2 = pd.DataFrame(test_list, columns=['study_id', 'experiment', 'experiment_name', 'year', 'participant',
#        'sex', 'role',  'dyad', 'species', 'session', 'trial', 'period', 'gain', 'enter', 'fight', 'latency',
#        'participant_2',
#        'sex_2', 'role_2', 'dyad_2', 'session_2', 'trial_2', 'gain_2', 'enter_2', 'fight_2', 'latency_2'])
# df_exp5_2=df_exp5_2.sort_values(by = ['dyad','session','trial'])

In [29]:
# studyID_standardized=df_exp5_2[['study_id', 'experiment', 'experiment_name', 'year', 'participant',
#        'sex', 'role', 'participant_2','sex_2', 'role_2', 'dyad', 'species', 
#        'session', 'trial', 'period', 'gain', 'enter', 'fight', 'latency',
#          'gain_2', 'enter_2', 'fight_2', 'latency_2']]

# comp_out_coherence_path = os.path.join(out_coherence_pathway, 'uher2008coherence_exp5_standardized_option2.csv')
# studyID_standardized.to_csv(comp_out_coherence_path, encoding='utf-8-sig', index=False)

# ##for one glossary
# names = studyID_standardized.columns.tolist()
# df = pd.DataFrame(names)
# df = df.rename(columns={0: "column_name"})
# df["description"] = ""
# studyID_glossary=df[["column_name", "description"]]

# comp_out_coherence_path_glossary = os.path.join(out_coherence_pathway, 'uher2008coherence_exp5_standardized_option2_glossary.csv')
# studyID_glossary.to_csv(comp_out_coherence_path_glossary, encoding='utf-8-sig', index=False)

In [30]:
##update to following files as per author request on 2025-10-10
complete_path_sav = os.path.join(original_data_pathway, "new original\BCGO_t1_t2_afternoon_obs_raw-data_NEW.sav")
sav_obs = pd.read_spss(complete_path_sav, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, 'BCGO_t1_t2_afternoon_obs_raw-data_NEW.csv')
# sav_obs.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)

sav_obs.columns = map(str.lower, sav_obs.columns) ##lowercase column names
sav_obs=sav_obs.applymap(lambda s: s.lower() if type(s) == str else s) ##lowercase entire df

sav_obs = sav_obs.assign(experiment='0')
sav_obs = sav_obs.assign(year='2005')
sav_obs = sav_obs.assign(study_id='uher2008coherence')
sav_obs.rename(columns={"sub": "participant",
                         'day':'observation_day', 
                         'unit':'instantaneous_sampling_unit', 
                        'prox':'o_proximity_to_other', 
                        'conta':'o_bodily_contact_with',
                        'give_gro':'o_gives_grooming_to', 
                        'rec_gro':'o_receives_grooming_from', 
                        's_groom':'o_self_grooming', 
                        's_care':'o_self_care', 
                        's_scra':'p_self_scratch', 
                        'nurs':'o_nursing', 
                        'al_play':'o_social_play',
                        'au_play':'o_auto_play', 
                        'obj_play':'o_object_play', 
                        'al_sex':'o_allo_sex', 
                        'au_sex':'o_auto_sex', 
                        'vigi':'o_vigilance', 
                        'feed':'o_feeding', 
                        'activ':'o_activity',
                        't':'period'}, inplace=True)
# sav_obs.columns

comp_path_name_errors = os.path.join(pathway_gen, "uher2008coherence_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
sav_obs['participant'] = sav_obs['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    sav_obs['participant'].replace(x, y, inplace=True)
# sav_obs['participant'].unique()

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
sav_obs= sav_obs.merge(apedf,left_on='participant', right_on='name', how='left')

In [31]:
sav_obs['period']=sav_obs['period'].astype(str)
sav_obs['experiment_name']=sav_obs.spec.str.cat(sav_obs.period, sep='_t')

experiment_obs_update = [['bonobo_t1.0','bonobo_t1_afternoon_observations'], 
                         ['bonobo_t2.0','bonobo_t2_afternoon_observations'], 
                         ['chimp_t1.0','chimpanzee_t1_afternoon_observations'], 
                         ['chimp_t2.0','chimpanzee_t2_afternoon_observations'],
                        ['gorilla_t1.0','gorilla_t1_afternoon_observations'], 
                        ['gorilla_t2.0','gorilla_t2_afternoon_observations'], 
                        ['orang_t1.0','orangutan_t1_afternoon_observations'], 
                        ['orang_t2.0','orangutan_t2_afternoon_observations']]
for x,y in experiment_obs_update:
    sav_obs['experiment_name'].replace(x, y, inplace=True, regex=True)
# sav_obs['experiment_name'].unique()
observation_dates = [
       ['bonobo_t1_afternoon_observations', '02',''],
       ['bonobo_t2_afternoon_obeservations', '02','03'],
       ['chimpanzee_t1_afternoon_observations', '02','03'],
       ['chimpanzee_t2_afternoon_obeservations', '02','03'],
       ['gorilla_t1_afternoon_observations','02',''],
      ['gorilla_t2_afternoon_obeservations', '02','03'],
       ['orangutan_t1_afternoon_observations','02','03'],
       ['orangutan_t2_afternoon_obeservations', '02', '03']]
for x,y,a in observation_dates:
       sav_obs.loc[sav_obs.experiment_name == x, ['month', 'day']] = y, a


complete_path_age = os.path.join(original_data_pathway, "table1.csv")
add_age = pd.read_csv(complete_path_age)   
sav_obs= sav_obs.merge(add_age,left_on='participant', right_on='name', how='left')
sav_obs.rename(columns={"age":"age_in_years"}, inplace=True)

comp_path_name_errors_obs = os.path.join(pathway_gen, "uher2008coherence_name_errors_obs.csv")
df_name_obs  = pd.read_csv(comp_path_name_errors_obs)

for x,y in zip(df_name_obs['wrong'],df_name_obs['right']):
    sav_obs['o_proximity_to_other'].replace(x, y, inplace=True)
    sav_obs['o_bodily_contact_with'].replace(x, y, inplace=True)
    sav_obs['o_gives_grooming_to'].replace(x, y, inplace=True)
    sav_obs['o_receives_grooming_from'].replace(x, y, inplace=True)
    sav_obs['o_nursing'].replace(x, y, inplace=True)
    sav_obs['o_social_play'].replace(x, y, inplace=True)

sav_obs['o_proximity_to_other'].replace('o-ra', 'raja', inplace=True)
sav_obs['o_bodily_contact_with'].replace('o-ra', 'raja', inplace=True)
sav_obs['o_gives_grooming_to'].replace('o-ra', 'raja', inplace=True)
sav_obs['o_receives_grooming_from'].replace('o-ra', 'raja', inplace=True)
sav_obs['o_nursing'].replace('o-ra', 'raja', inplace=True)
sav_obs['o_social_play'].replace('o-ra', 'raja', inplace=True)

sav_obs['o_proximity_to_other'].replace(220.0, 'flynn', inplace=True)
sav_obs['o_proximity_to_other'].replace(9.0, 'observer', inplace=True)

sav_obs['o_proximity_to_other'].unique()

array([0.0, 'yasa', 'luiza', 'kuno', 'limbuko', 'joey', 'me', 'ulindi',
       'observer', nan, 'corrie', 'lobo', 'riet', 'tai', 'ulla', 'robert',
       'fraukje', 'fifi', 'lome', 'natascha', 'flynn', 'pia', 'dorien',
       'brent', 'jahaga', 'frodo', 'patrick', 'sandra', 'fynn',
       'gertruida', 'kibara', 'viringika', "ruby_b'windi", 'n’diki',
       'bebe', 'gorgo', 'pagai', 'raja', 'kila', 'dunja', 'padana',
       'dokana', 'bimbo', 'pini'], dtype=object)

In [32]:
studyID_standardized=sav_obs[['study_id','experiment','experiment_name', 'year', 'month', 'day',
                              'participant','age_in_years', 'sex','species',
                              'period','observation_day', 'instantaneous_sampling_unit',
        'o_proximity_to_other', 'o_bodily_contact_with',
       'o_gives_grooming_to', 'o_receives_grooming_from', 'o_self_grooming',
       'o_self_care', 'p_self_scratch', 'o_nursing', 'o_social_play',
       'o_auto_play', 'o_object_play', 'o_allo_sex', 'o_auto_sex',
       'o_vigilance', 'o_feeding', 'o_activity', 
        ]]

comp_out_coherence_path_stand = os.path.join(out_coherence_pathway, 'uher2008coherence_observations_afternoon_standardized.csv')
studyID_standardized.to_csv(comp_out_coherence_path_stand, encoding='utf-8-sig', index=False)

##for one glossary
names = studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_coherence_pathway, 'uher2008coherence_observations_afternoon_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

In [33]:
complete_path_exp2 = os.path.join(original_data_pathway, r"new original\uher2008coherence_exp2_standardized_UPDATE.csv")
exp2 = pd.read_csv(complete_path_exp2)
exp2.rename(columns={"Unnamed: 13":"remove_list"}, inplace=True)
exp2['remove_list']=exp2['remove_list'].astype(str)
exp2 = exp2[~exp2.remove_list.str.contains("delete - he did not come in")]


In [34]:
studyID_standardized=exp2[['study_id', 'experiment', 'experiment_name', 'year', 'participant',
       'age_in_years', 'sex', 'species', 'session', 'period',
       'duration_of_proximity_to_experimenter',
       'freq_of_quasi-aggressive_acts', 'food_taken']]
comp_out_coherence_path_stand = os.path.join(out_coherence_pathway, 'uher2008coherence_exp2_standardized.csv')
studyID_standardized.to_csv(comp_out_coherence_path_stand, encoding='utf-8-sig', index=False)

##for one glossary
names = studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_coherence_pathway, 'uher2008coherence_exp2_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)



In [35]:
complete_path_exp6 = os.path.join(original_data_pathway, r"new original\uher2008coherence_exp6_standardized_UPDATE.csv")
exp6 = pd.read_csv(complete_path_exp6)
# exp6.columns

In [36]:
studyID_standardized = exp6[['study_id', 'experiment', 'experiment_name', 'year', 'participant',
       'age_in_years', 'sex', 'species', 'session', 'period', 'activity',
       'latency_to_reach_or_touch', 'not_found_or_taken',
       'percentage_items_found', 'food_item']]
comp_out_coherence_path_stand = os.path.join(out_coherence_pathway, 'uher2008coherence_exp6_standardized.csv')
studyID_standardized.to_csv(comp_out_coherence_path_stand, encoding='utf-8-sig', index=False)

##for one glossary
names = studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_coherence_pathway, 'uher2008coherence_exp6_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

In [37]:
complete_path_exp11 = os.path.join(original_data_pathway, r"new original\uher2008coherence_exp11_standardized_UPDATE.csv")
exp11 = pd.read_csv(complete_path_exp11)
# exp11.columns

In [38]:
studyID_standardized = exp11[['study_id', 'experiment', 'experiment_name', 'year', 'participant',
       'age_in_years', 'sex', 'species', 'session', 'period',
       'latency_to_reach_or_touch', 'duration_of_action_with_object']]
comp_out_coherence_path_stand = os.path.join(out_coherence_pathway, 'uher2008coherence_exp11_standardized.csv')
studyID_standardized.to_csv(comp_out_coherence_path_stand, encoding='utf-8-sig', index=False)

##for one glossary
names = studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_coherence_pathway, 'uher2008coherence_exp11_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)